# Optimized SFT and DPO Training Pipeline with Unsloth

This notebook demonstrates:
1. Fine-tuning a Llama-3.2-1B model using Supervised Fine-Tuning (SFT) with Unsloth
2. Evaluating the SFT model against the base model
3. Performing DPO training on the SFT model
4. Evaluating the DPO model against the base model

## Setup
First, let's create a directory for logs and install the necessary packages.

In [ ]:
# Create logs directory
!mkdir -p ./outputs/logs

# Start log
!echo "=== STARTING OPTIMIZED SFT AND DPO RUN ===" > ./outputs/logs/training.log

In [ ]:
# Install required packages
!pip install unsloth transformers trl peft datasets torch accelerate bitsandbytes wandb

In [ ]:
# Verify Python and key dependencies
!python --version
!pip list | grep -E "unsloth|transformers|torch|trl|peft|datasets|accelerate|bitsandbytes"

## Import Libraries
Let's import all the necessary libraries for our training.

In [ ]:
import time
start_time = time.time()
print("✅ [CHECKPOINT] Python execution started")

import os
import random
import json
import numpy as np
from collections import defaultdict

try:
    from unsloth.chat_templates import get_chat_template
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTTrainer, DPOTrainer
    from peft import PeftModel
    import torch
    from datasets import load_dataset
    from transformers import (
        AutoTokenizer, 
        TrainingArguments, 
        TextStreamer,
        AutoModelForCausalLM,
    )
    print("✅ [CHECKPOINT] Imports successful")
except ImportError as e:
    print(f"❌ ImportError: {e}")
    raise

## Check CUDA Availability
Let's verify if CUDA is available for GPU acceleration.

In [ ]:
# Check CUDA
print("CUDA available:", torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if torch.cuda.is_available():
    print("GPU Info:")
    print(f"- Device count: {torch.cuda.device_count()}")
    print(f"- Current device: {torch.cuda.current_device()}")
    print(f"- Device name: {torch.cuda.get_device_name(0)}")

## Set Random Seeds
For reproducibility, we'll set random seeds for all libraries.

In [ ]:
# Seed for reproducibility
def set_seed(seed=1):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(1)
print("✅ [CHECKPOINT] Seed set")

## Load Model
Now let's load the base model using Unsloth's optimized loading.

In [ ]:
# Load model via Unsloth
print("Loading model - this may take a moment...")
model_load_start = time.time()

max_seq_length = 1024
dtype = None
load_in_4bit = True

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        dtype=dtype,
    )
    print(f"✅ [CHECKPOINT] Model loaded in {time.time() - model_load_start:.2f}s")
    print("Model config:", model.config)
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    raise

## Setup PEFT/LoRA
Configure the model for Parameter-Efficient Fine-Tuning using LoRA.

In [ ]:
# Set up PEFT with LoRA
peft_start = time.time()
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_rslora=True,
    use_gradient_checkpointing=True
)
print(f"✅ [CHECKPOINT] PEFT model created in {time.time() - peft_start:.2f}s")

## Prepare Dataset
Let's prepare the Alpaca dataset for fine-tuning.

In [ ]:
# Prepare dataset formatting function
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

In [ ]:
# Load and process the dataset
dataset_start = time.time()
print("Loading dataset...")

try:
    dataset = load_dataset("yahma/alpaca-cleaned", split="train[:500]")
    dataset = dataset.map(formatting_prompts_func, batched=True)
    print(f"✅ [CHECKPOINT] Dataset loaded and processed in {time.time() - dataset_start:.2f}s with {len(dataset)} examples")
    
    # Display a sample
    print("\nSample from dataset:")
    print(dataset[0]["text"][:500] + "...")
except Exception as e:
    print(f"❌ Failed to load/process dataset: {e}")
    raise

## Configure Training
Now let's set up the training configuration using SFTTrainer.

In [ ]:
# Configure the SFT Trainer
output_dir = "./outputs"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    dataset_num_proc=4,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1.0,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=output_dir,
        report_to="none",
        save_strategy="steps",
        save_steps=50,
        gradient_checkpointing=True,
        max_grad_norm=1.0,
        dataloader_num_workers=1,
        dataloader_pin_memory=True,
    ),
)

print("✅ [CHECKPOINT] Trainer configured, starting training...")

## Train the Model
Let's start the training process.

In [ ]:
# Train the model
train_start = time.time()

try:
    trainer_stats = trainer.train()
    print(f"✅ [CHECKPOINT] Training completed in {time.time() - train_start:.2f}s")
    print(f"Training stats: {trainer_stats}")
except Exception as e:
    print(f"❌ Training failed: {e}")
    raise

## Save the Model
Save the trained LoRA adapter.

In [ ]:
# Save adapter
save_start = time.time()
sft_adapter_path = "./outputs/sft_lora_adapter"

try:
    model.save_pretrained(
        sft_adapter_path,
        save_adapter=True,
        save_config=True
    )
    print(f"✅ [CHECKPOINT] SFT model adapter saved in {time.time() - save_start:.2f}s")
except Exception as e:
    print(f"❌ Failed to save model: {e}")
    raise

## Test the Fine-Tuned Model
Let's test our fine-tuned model with a sample prompt.

In [ ]:
# Test the fine-tuned model
print("Testing fine-tuned model with a sample prompt...")

try:
    # Load the adapter onto a fresh model instance
    test_model, test_tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit
    )
    
    # Load the PEFT adapter
    test_model = PeftModel.from_pretrained(test_model, sft_adapter_path)
    
    # Create a text streamer for nice output
    streamer = TextStreamer(test_tokenizer, skip_prompt=True)
    
    # Sample prompt
    test_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Explain the concept of fine-tuning in machine learning.

### Response:
"""
    
    print("\nPrompt:\n", test_prompt)
    print("\nGenerated Response:")
    
    # Generate response
    inputs = test_tokenizer(test_prompt, return_tensors="pt").to(device)
    
    output = test_model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
        streamer=streamer
    )
except Exception as e:
    print(f"❌ Testing failed: {e}")

## Set up Evaluation
Now we'll set up a comprehensive evaluation comparing our fine-tuned model against the base model using a judge model.

In [ ]:
# Set up evaluation
eval_start = time.time()

print("Loading models for evaluation...")

def load_base_model():
    print("Loading base model")
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
            max_seq_length=1024,
            load_in_4bit=True,
            dtype=None,
        )
        return model, tokenizer
    except Exception as e:
        print(f"❌ Failed to load base model: {e}")
        raise

def load_sft_model():
    print("Loading SFT model")
    try:
        base_model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
            max_seq_length=1024,
            load_in_4bit=True,
            dtype=None,
        )
        peft_model = PeftModel.from_pretrained(base_model, sft_adapter_path)
        return peft_model, tokenizer
    except Exception as e:
        print(f"❌ Failed to load SFT model: {e}")
        raise

def load_judge_model():
    print("Loading judge model")
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/Llama-3-8B-unsloth-bnb-4bit",
            max_seq_length=1024,
            load_in_4bit=True,
            dtype=None,
        )
        return model, tokenizer
    except Exception as e:
        print(f"❌ Failed to load judge model: {e}")
        raise

try:
    base_model, base_tokenizer = load_base_model()
    sft_model, sft_tokenizer = load_sft_model()
    judge_model, judge_tokenizer = load_judge_model()
    print("✅ [CHECKPOINT] Evaluation models loaded")
except Exception as e:
    print(f"❌ Failed to load evaluation models: {e}")
    raise

In [ ]:
# Define helper functions for evaluation
def format_prompt(instruction):
    return f"""Below is an instruction that describes a task. Write a response that completes the request.

### Instruction:
{instruction}

### Response:"""

def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=1024,
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_response.replace(prompt, "").strip()

def get_judgment(instruction, base_response, sft_response):
    prompt = f"""You are an impartial judge evaluating two responses to an instruction. Score based on:
* Instruction Following: How well the response addresses the instruction.
* Helpfulness: Usefulness and practicality of the response.
* Accuracy: Factual correctness and logical consistency.
* Clarity: Readability and coherence.
* Detail: Appropriate level of detail without redundancy.

Return ONLY a single number: 
1 = First response (Base Model) is significantly better
2 = Second response (SFT Model) is significantly better
0 = Both are comparable (similar quality or minor differences)

Instruction:
{instruction}

First Response (Base Model):
{base_response}

Second Response (SFT Model):
{sft_response}

Judgment (1/2/0):"""
    
    inputs = judge_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to("cuda")
    
    with torch.no_grad():
        outputs = judge_model.generate(
            **inputs,
            max_new_tokens=3,
            temperature=0.0,
            do_sample=False,
        )
    
    verdict = judge_tokenizer.decode(outputs[0], skip_special_tokens=True)
    last_char = verdict.strip()[-1] if verdict.strip() else '0'
    return int(last_char) if last_char.isdigit() else 0

## Run Comparative Evaluation
Let's run a comparative evaluation between the base model and our fine-tuned model.

In [ ]:
# Load evaluation dataset
print("Loading evaluation dataset...")
MAX_SAMPLES = 20

try:
    eval_dataset = load_dataset("tatsu-lab/alpaca_eval", "alpaca_eval", trust_remote_code=True)["eval"]
    eval_dataset = eval_dataset.shuffle(seed=42).select(range(min(len(eval_dataset), MAX_SAMPLES)))
    print(f"Loaded evaluation dataset with {len(eval_dataset)} examples")
except Exception as e:
    print(f"❌ Failed to load evaluation dataset: {e}")
    raise

In [ ]:
# Run evaluation
print(f"Running evaluation on {len(eval_dataset)} examples...")
sft_results = []

for i, example in enumerate(eval_dataset):
    try:
        print(f"\nExample {i+1}/{len(eval_dataset)}")
        print(f"Evaluating: {example['instruction'][:50]}...")
        
        prompt = format_prompt(example["instruction"])
        base_response = generate_response(base_model, base_tokenizer, prompt)
        sft_response = generate_response(sft_model, sft_tokenizer, prompt)
        
        if base_response.strip() == sft_response.strip():
            verdict = 0
            print("Responses identical - verdict: 0 (tie)")
        else:
            verdict = get_judgment(example["instruction"], base_response, sft_response)
            print(f"Verdict: {verdict} ({['Base wins', 'Tie', 'SFT wins'][verdict]})")
        
        sft_results.append({
            "instruction": example["instruction"],
            "base_response": base_response,
            "sft_response": sft_response,
            "verdict": verdict
        })
    except Exception as e:
        print(f"Error processing example: {e}")
        continue

print(f"✅ [CHECKPOINT] SFT evaluation completed in {time.time() - eval_start:.2f}s")

## Analyze SFT Results
Let's analyze and save the evaluation results.

In [ ]:
# Calculate statistics
verdicts = [r["verdict"] for r in sft_results]
sft_stats = {
    "base_wins": verdicts.count(1),
    "sft_wins": verdicts.count(2),
    "ties": verdicts.count(0),
    "total": len(verdicts),
    "sft_win_rate": verdicts.count(2) / len(verdicts) if verdicts else 0,
}

# Print summary statistics
print("\n=== SFT Evaluation Summary ===")
print(f"SFT Win Rate: {sft_stats['sft_win_rate']:.1%}")
print(f"Base Wins: {sft_stats['base_wins']} | SFT Wins: {sft_stats['sft_wins']} | Ties: {sft_stats['ties']}")

In [ ]:
# Save results
sft_output_file = "./outputs/alpacaeval_sft_results.json"

with open(sft_output_file, "w") as f:
    json.dump({
        "statistics": sft_stats,
        "results": sft_results,
        "config": {
            "base_model": "unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
            "sft_adapter": "/output/sft_lora_adapter",
            "judge_model": "unsloth/Llama-3.2-8B-unsloth-bnb-4bit",
            "num_samples": len(sft_results)
        }
    }, f, indent=2)

print(f"✅ [CHECKPOINT] SFT results saved to {sft_output_file}")

## Prepare for DPO Training
Now we'll prepare for Direct Preference Optimization (DPO) training using the SFT model as our starting point.

In [ ]:
# Load preference dataset for DPO
print("Loading preference dataset for DPO...")

try:
    # We'll use a synthetic preference dataset for demonstration
    # In practice, you would use your own preference dataset with chosen/rejected pairs
    dpo_dataset = load_dataset("Intel/orca_dpo_pairs", split="train[:100]")
    
    # Format the dataset for DPO
    def format_dpo_example(example):
        return {
            "prompt": example["question"],
            "chosen": example["chosen"],
            "rejected": example["rejected"],
        }
    
    dpo_dataset = dpo_dataset.map(format_dpo_example)
    print(f"Loaded DPO dataset with {len(dpo_dataset)} examples")
    
    # Display a sample
    print("\nSample from DPO dataset:")
    print("Prompt:", dpo_dataset[0]["prompt"][:100] + "...")
    print("Chosen response:", dpo_dataset[0]["chosen"][:100] + "...")
    print("Rejected response:", dpo_dataset[0]["rejected"][:100] + "...")
except Exception as e:
    print(f"❌ Failed to load/process DPO dataset: {e}")
    raise

## Prepare DPO Models
We need to prepare the SFT model and a reference model for DPO training.

In [ ]:
# Load the SFT model as our policy model
print("Loading SFT model for DPO training...")

try:
    # Load base model
    dpo_model, dpo_tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
        max_seq_length=max_seq_length,
        load_in_4bit=True,
        dtype=None,
    )
    
    # Load the SFT adapter
    dpo_model = PeftModel.from_pretrained(dpo_model, sft_adapter_path)
    
    # Load the base model as reference model
    ref_model, _ = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
        max_seq_length=max_seq_length,
        load_in_4bit=True,
        dtype=None,
    )
    
    print("✅ [CHECKPOINT] DPO models loaded")
except Exception as e:
    print(f"❌ Failed to load DPO models: {e}")
    raise

## Configure DPO Training
Set up the DPO trainer with appropriate parameters.

In [ ]:
# Configure DPO trainer
dpo_output_dir = "./outputs/dpo_model"

# Define DPO training arguments
dpo_training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=1.0,
    learning_rate=5e-6,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir=dpo_output_dir,
    report_to="none",
    save_strategy="steps",
    save_steps=50,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    dataloader_num_workers=1,
    dataloader_pin_memory=True,
)

# Create DPO trainer
dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=ref_model,
    args=dpo_training_args,
    beta=0.1,  # The beta parameter for DPO loss
    train_dataset=dpo_dataset,
    tokenizer=dpo_tokenizer,
    max_length=max_seq_length,
    max_prompt_length=512,
    generate_during_eval=True,
)

print("✅ [CHECKPOINT] DPO trainer configured")

## Run DPO Training
Perform DPO training on the SFT model.

In [ ]:
# Train with DPO
print("Starting DPO training...")
dpo_train_start = time.time()

try:
    dpo_trainer.train()
    print(f"✅ [CHECKPOINT] DPO training completed in {time.time() - dpo_train_start:.2f}s")
except Exception as e:
    print(f"❌ DPO training failed: {e}")
    raise

## Save DPO Model
Save the DPO-trained model adapter.

In [ ]:
# Save DPO adapter
dpo_adapter_path = "./outputs/dpo_lora_adapter"

try:
    dpo_trainer.model.save_pretrained(
        dpo_adapter_path,
        save_adapter=True,
        save_config=True
    )
    print(f"✅ [CHECKPOINT] DPO model adapter saved to {dpo_adapter_path}")
except Exception as e:
    print(f"❌ Failed to save DPO model: {e}")
    raise

## Evaluate DPO Model
Now let's evaluate the DPO model against the base model using the same evaluation framework.

In [ ]:
# Load DPO model for evaluation
print("Loading DPO model for evaluation...")

try:
    dpo_eval_model, dpo_eval_tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
        max_seq_length=1024,
        load_in_4bit=True,
        dtype=None,
    )
    
    dpo_eval_model = PeftModel.from_pretrained(dpo_eval_model, dpo_adapter_path)
    print("✅ [CHECKPOINT] DPO evaluation model loaded")
except Exception as e:
    print(f"❌ Failed to load DPO evaluation model: {e}")
    raise

In [ ]:
# Run DPO evaluation
print(f"Running DPO evaluation on {len(eval_dataset)} examples...")
dpo_results = []

for i, example in enumerate(eval_dataset):
    try:
        print(f"\nExample {i+1}/{len(eval_dataset)}")
        print(f"Evaluating: {example['instruction'][:50]}...")
        
        prompt = format_prompt(example["instruction"])
        base_response = generate_response(base_model, base_tokenizer, prompt)
        dpo_response = generate_response(dpo_eval_model, dpo_eval_tokenizer, prompt)
        
        if base_response.strip() == dpo_response.strip():
            verdict = 0
            print("Responses identical - verdict: 0 (tie)")
        else:
            verdict = get_judgment(example["instruction"], base_response, dpo_response)
            print(f"Verdict: {verdict} ({['Base wins', 'Tie', 'DPO wins'][verdict]})")
        
        dpo_results.append({
            "instruction": example["instruction"],
            "base_response": base_response,
            "dpo_response": dpo_response,
            "verdict": verdict
        })
    except Exception as e:
        print(f"Error processing example: {e}")
        continue

print(f"✅ [CHECKPOINT] DPO evaluation completed in {time.time() - eval_start:.2f}s")

## Analyze DPO Results
Compare the DPO model performance against the base model.

In [ ]:
# Calculate DPO statistics
dpo_verdicts = [r["verdict"] for r in dpo_results]
dpo_stats = {
    "base_wins": dpo_verdicts.count(1),
    "dpo_wins": dpo_verdicts.count(2),
    "ties": dpo_verdicts.count(0),
    "total": len(dpo_verdicts),
    "dpo_win_rate": dpo_verdicts.count(2) / len(dpo_verdicts) if dpo_verdicts else 0,
}

# Print summary statistics
print("\n=== DPO Evaluation Summary ===")
print(f"DPO Win Rate: {dpo_stats['dpo_win_rate']:.1%}")
print(f"Base Wins: {dpo_stats['base_wins']} | DPO Wins: {dpo_stats['dpo_wins']} | Ties: {dpo_stats['ties']}")

In [ ]:
# Save DPO results
dpo_output_file = "./outputs/alpacaeval_dpo_results.json"

with open(dpo_output_file, "w") as f:
    json.dump({
        "statistics": dpo_stats,
        "results": dpo_results,
        "config": {
            "base_model": "unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
            "dpo_adapter": "/output/dpo_lora_adapter",
            "judge_model": "unsloth/Llama-3.2-8B-unsloth-bnb-4bit",
            "num_samples": len(dpo_results)
        }
    }, f, indent=2)

print(f"✅ [CHECKPOINT] DPO results saved to {dpo_output_file}")

## Compare SFT and DPO Results
Let's compare the performance of SFT and DPO models.

In [ ]:
print("\n=== Model Comparison Summary ===")
print(f"SFT Win Rate: {sft_stats['sft_win_rate']:.1%}")
print(f"DPO Win Rate: {dpo_stats['dpo_win_rate']:.1%}")
print(f"\nSFT Performance: Base Wins: {sft_stats['base_wins']} | SFT Wins: {sft_stats['sft_wins']} | Ties: {sft_stats['ties']}")
print(f"DPO Performance: Base Wins: {dpo_stats['base_wins']} | DPO Wins: {dpo_stats['dpo_wins']} | Ties: {dpo_stats['ties']}")

# Calculate improvement
improvement = (dpo_stats['dpo_win_rate'] - sft_stats['sft_win_rate']) * 100
print(f"\nDPO improvement over SFT: {improvement:.1f} percentage points")

## Final Summary
Print a final summary of the entire pipeline.

In [ ]:
print("\n=== FINAL SUMMARY ===")
print(f"SFT Win Rate: {sft_stats['sft_win_rate']:.1%}")
print(f"DPO Win Rate: {dpo_stats['dpo_win_rate']:.1%}")
print(f"DPO Improvement: {improvement:.1f} percentage points")
print(f"\nFULL PIPELINE COMPLETED SUCCESSFULLY in {time.time() - start_time:.2f}s")